In [ ]:
# @title Saved-model bank loader

# 6 parameters:
#   1) save_root
#   2) run_group
#   3) dataset_name      in {"cifar10", "cifar100"}
#   4) model_name        in {"resnet18_pgd", "resnet18_trades", "resnet18_mart"}
#   5) seed              int
#   6) checkpoint_tag    in {"best", "last"}

import re
import json
from pathlib import Path

import numpy as np
import jax
import jax.numpy as jnp
from jax import random
import flax
import flax.linen as nn
from flax import serialization

def natural_key(s: str):
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", s)]

def parse_model_name(model_name: str):
    if not model_name.startswith("resnet18_"):
        raise ValueError(f"Unexpected model_name: {model_name}")
    return "resnet18", model_name.split("_", 1)[1]

def run_dir(save_root, run_group, dataset_name, model_name, seed):
    return Path(save_root) / run_group / dataset_name / model_name / f"seed_{seed:02d}"

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

DATASET_CFG = {
    "cifar10": {"num_classes": 10, "image_size": 32},
    "cifar100": {"num_classes": 100, "image_size": 32},
}

class ResidualBlock(nn.Module):
    features: int
    stride: int = 1

    @nn.compact
    def __call__(self, x, train: bool):
        residual = x

        y = nn.Conv(self.features, (3, 3), strides=(self.stride, self.stride),
                    padding="SAME", use_bias=False)(x)
        y = nn.BatchNorm(use_running_average=not train, momentum=0.9, epsilon=1e-5)(y)
        y = nn.relu(y)

        y = nn.Conv(self.features, (3, 3), strides=(1, 1),
                    padding="SAME", use_bias=False)(y)
        y = nn.BatchNorm(use_running_average=not train, momentum=0.9, epsilon=1e-5,
                         scale_init=nn.initializers.zeros)(y)

        if residual.shape != y.shape:
            residual = nn.Conv(self.features, (1, 1), strides=(self.stride, self.stride),
                               use_bias=False)(residual)
            residual = nn.BatchNorm(use_running_average=not train, momentum=0.9, epsilon=1e-5)(residual)

        return nn.relu(residual + y)

class ResNet18(nn.Module):
    num_classes: int
    image_size: int = 32

    @nn.compact
    def __call__(self, x, train: bool):
        x = nn.Conv(64, (3, 3), strides=(1, 1), padding="SAME",
                    use_bias=False, name="stem_conv")(x)
        x = nn.BatchNorm(use_running_average=not train, momentum=0.9, epsilon=1e-5,
                         name="stem_bn")(x)
        x = nn.relu(x)
        self.sow("intermediates", "act_stem", x)

        block_specs = [
            (64, 1), (64, 1),
            (128, 2), (128, 1),
            (256, 2), (256, 1),
            (512, 2), (512, 1),
        ]

        for i, (features, stride) in enumerate(block_specs, start=1):
            x = ResidualBlock(features=features, stride=stride, name=f"block{i}")(x, train=train)
            self.sow("intermediates", f"act_block{i}", x)

        x = jnp.mean(x, axis=(1, 2))
        self.sow("intermediates", "act_pre_logits", x)
        logits = nn.Dense(self.num_classes, name="head")(x)
        return logits

def build_model(model_name: str, dataset_name: str):
    arch, defense = parse_model_name(model_name)
    if arch != "resnet18":
        raise ValueError(model_name)

    num_classes = DATASET_CFG[dataset_name]["num_classes"]
    model = ResNet18(num_classes=num_classes, image_size=32)
    model_cfg = {
        "model_name": model_name,
        "arch": arch,
        "defense_method": defense,
        "num_classes": num_classes,
        "image_size": 32,
    }
    return model, model_cfg

def load_rep_bundle(save_root, run_group, dataset_name, model_name, seed, checkpoint_tag="best"):
    if checkpoint_tag not in ("best", "last"):
        raise ValueError("checkpoint_tag must be 'best' or 'last'")

    rdir = run_dir(save_root, run_group, dataset_name, model_name, seed)
    meta_path = rdir / "meta.json"
    ckpt_path = rdir / f"{checkpoint_tag}_analysis.msgpack"

    if not meta_path.exists():
        raise FileNotFoundError(meta_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(ckpt_path)

    meta = load_json(meta_path)
    model, model_cfg = build_model(model_name, dataset_name)

    image_size = meta["dataset_cfg"]["image_size"]
    dummy = jnp.zeros((1, image_size, image_size, 3), dtype=jnp.float32)
    key = random.PRNGKey(0)

    init_vars = model.init({"params": key}, dummy, train=False)

    template = {
        "params": init_vars["params"],
        "batch_stats": init_vars.get("batch_stats", {}),
        "step": np.asarray(0, dtype=np.int32),
    }

    with open(ckpt_path, "rb") as f:
        payload = serialization.from_bytes(template, f.read())

    variables = {"params": payload["params"]}
    if payload["batch_stats"]:
        variables["batch_stats"] = payload["batch_stats"]

    layer_names = meta["layer_names"]

    def apply_logits(x, train=False):
        return model.apply(variables, x, train=train)

    def get_activations(x, train=False):
        logits, mut = model.apply(variables, x, train=train, mutable=["intermediates"])
        acts = {k: v[0] for k, v in mut["intermediates"].items()}
        return logits, acts

    def get_layer(x, layer_name, train=False):
        _, acts = get_activations(x, train=train)
        return acts[layer_name]

    def jvp_logits(x, v):
        f = lambda z: model.apply(variables, z, train=False)
        return jax.jvp(f, (x,), (v,))

    def jvp_layer(x, v, layer_name):
        def f(z):
            _, acts = get_activations(z, train=False)
            return acts[layer_name]
        return jax.jvp(f, (x,), (v,))

    bundle = {
        "run_dir": str(rdir),
        "meta": meta,
        "model_cfg": model_cfg,
        "model": model,
        "variables": variables,
        "step": int(payload["step"]),
        "layer_names": layer_names,
        "apply_logits": apply_logits,
        "get_activations": get_activations,
        "get_layer": get_layer,
        "jvp_logits": jvp_logits,
        "jvp_layer": jvp_layer,
    }
    return bundle

print("Loader ready.")
print("Use:")
print("load_rep_bundle(save_root, run_group, dataset_name, model_name, seed, checkpoint_tag='best')")

In [ ]:
from google.colab import drive
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
drive.mount("/content/drive", force_remount=True)


In [ ]:
# @title 1. Setup and experiment config

!pip install -q flax tensorflow tensorflow-datasets scipy pandas seaborn scikit-learn

import os
import gc
import re
import json
import math
import itertools
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score

import jax
import jax.numpy as jnp
from jax import random
import tensorflow as tf
import tensorflow_datasets as tfds

try:
    import jax.tools.colab_tpu
    jax.tools.colab_tpu.setup_tpu()
except Exception:
    pass

print("JAX backend:", jax.default_backend())
print("Devices:", jax.local_device_count())

sns.set_context("talk")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300

# ============================================================
# ROOTS
# ============================================================
SAVE_ROOT = "/content/drive/MyDrive/representation_bank"
CLEAN_RUN_GROUP = "repbank_clean_v1"
ADV_RUN_GROUP = "adv_resnet18_cifar_v1"

# ============================================================
# MAIN CONFIG
# ============================================================
GLOBAL_SEED = 0
rng = np.random.default_rng(GLOBAL_SEED)

DATASET_NAME = "cifar10"   # change to "cifar100" later
CHECKPOINT_TAG = "best"

# model groups
STANDARD_MODEL_NAME = "resnet18"
ADV_MODEL_NAMES = ["resnet18_pgd", "resnet18_trades", "resnet18_mart"]
STANDARD_SEEDS = list(range(10))
ADV_SEEDS = list(range(5))

# layers
MAIN_LAYER = "act_pre_logits"
LAYER_ABLATION_LAYERS = ["act_block6", "act_block8", "act_pre_logits"]

# learned family
K_MAX = 64
K_LIST = [64, 32, 16, 8]
# finite-difference coefficient for tangent estimation wrt flow basis
FD_EPS = 0.5
# base smooth flow bank:
# use all scalar sine modes on a 6x6 grid -> 36 scalar modes -> 72 dx/dy flows
MAX_U = 6
MAX_V = 6
MODE_LIST = [(u, v) for u in range(1, MAX_U + 1) for v in range(1, MAX_V + 1)]
N_SCALAR_MODES = len(MODE_LIST)      # 36
N_BROAD_FLOWS = 2 * N_SCALAR_MODES   # 72

# family-learning split
N_FAM_LEARN = 500

# geometry split for G estimation and CKA
N_GEOM = 2000

# benchmark image selection
N_BENCH_CANDIDATES = 256
N_BENCH_IMAGES = 128
MIN_CLEAN_CORRECT_COUNT = 12

# discovery/test splits per comparison
N_SPLITS = 10
DISC_FRAC = 0.6   # for matched groups of size 5 => 3 discovery / 2 test

# diagnostic probes
PROBES_PER_SIDE = 2     # 2 A-favoring + 2 B-favoring
N_RANDOM_CAND = 128
TOP_POOLED_EIGS = 12

# finite perturbation amplitudes in RMS flow pixels
FLOW_RMS_AMPS = np.array([0.5, 1.0], dtype=np.float32)

# layer ablation K
LAYER_ABLATION_K = 16

# batching
FEATURE_BATCH_SIZE = 256
LOGIT_BATCH_SIZE = 256
GRAM_BATCH_SIZE_TOTAL = 32

# stats
N_BOOT = 500

# ============================================================
# DEBUG SWITCH
# ============================================================
DEBUG = False
if DEBUG:
    N_FAM_LEARN = 128
    N_GEOM = 256
    N_BENCH_CANDIDATES = 64
    N_BENCH_IMAGES = 24
    MIN_CLEAN_CORRECT_COUNT = 8
    N_SPLITS = 4
    N_BOOT = 100

# ============================================================
# OUTPUT DIRS
# ============================================================
ROOT_DIR = "/content/drive/MyDrive/sras_adv_resnet18_experiment"
RUN_TAG = f"{DATASET_NAME}_standard_adv_resnet18_local_geometry"
RUN_DIR = os.path.join(ROOT_DIR, RUN_TAG)
ARRAY_DIR = os.path.join(RUN_DIR, "arrays")
TABLE_DIR = os.path.join(RUN_DIR, "tables")
FIG_DIR = os.path.join(RUN_DIR, "figures")

for d in [RUN_DIR, ARRAY_DIR, TABLE_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Saving to:", RUN_DIR)

In [ ]:
# @title 2. Unified loader for clean + adversarial ResNet-18 banks

import flax
import flax.linen as nn
from flax import serialization

def natural_key(s: str):
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", s)]

def run_dir(save_root, run_group, dataset_name, model_name, seed):
    return Path(save_root) / run_group / dataset_name / model_name / f"seed_{seed:02d}"

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

DATASET_CFG = {
    "cifar10": {
        "num_classes": 10,
        "image_size": 32,
        "mean": (0.4914, 0.4822, 0.4465),
        "std":  (0.2470, 0.2435, 0.2616),
    },
    "cifar100": {
        "num_classes": 100,
        "image_size": 32,
        "mean": (0.5071, 0.4867, 0.4408),
        "std":  (0.2675, 0.2565, 0.2761),
    },
}

def parse_model_name(model_name: str):
    if model_name == "resnet18":
        return "resnet18", "standard"
    if model_name.startswith("resnet18_"):
        return "resnet18", model_name.split("_", 1)[1]
    raise ValueError(f"Unexpected model_name: {model_name}")

class ResidualBlock(nn.Module):
    features: int
    stride: int = 1

    @nn.compact
    def __call__(self, x, train: bool):
        residual = x

        y = nn.Conv(
            self.features, (3, 3),
            strides=(self.stride, self.stride),
            padding="SAME",
            use_bias=False
        )(x)
        y = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5
        )(y)
        y = nn.relu(y)

        y = nn.Conv(
            self.features, (3, 3),
            strides=(1, 1),
            padding="SAME",
            use_bias=False
        )(y)
        y = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5,
            scale_init=nn.initializers.zeros
        )(y)

        if residual.shape != y.shape:
            residual = nn.Conv(
                self.features, (1, 1),
                strides=(self.stride, self.stride),
                use_bias=False
            )(residual)
            residual = nn.BatchNorm(
                use_running_average=not train,
                momentum=0.9,
                epsilon=1e-5
            )(residual)

        return nn.relu(residual + y)

class ResNet18(nn.Module):
    num_classes: int
    image_size: int = 32

    @nn.compact
    def __call__(self, x, train: bool):
        x = nn.Conv(
            64, (3, 3),
            strides=(1, 1),
            padding="SAME",
            use_bias=False,
            name="stem_conv"
        )(x)
        x = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5,
            name="stem_bn"
        )(x)
        x = nn.relu(x)
        self.sow("intermediates", "act_stem", x)

        block_specs = [
            (64, 1), (64, 1),
            (128, 2), (128, 1),
            (256, 2), (256, 1),
            (512, 2), (512, 1),
        ]

        for i, (features, stride) in enumerate(block_specs, start=1):
            x = ResidualBlock(features=features, stride=stride, name=f"block{i}")(x, train=train)
            self.sow("intermediates", f"act_block{i}", x)

        x = jnp.mean(x, axis=(1, 2))
        self.sow("intermediates", "act_pre_logits", x)
        logits = nn.Dense(self.num_classes, name="head")(x)
        return logits

def build_model(model_name: str, dataset_name: str):
    arch, defense = parse_model_name(model_name)
    if arch != "resnet18":
        raise ValueError(model_name)

    num_classes = DATASET_CFG[dataset_name]["num_classes"]
    model = ResNet18(num_classes=num_classes, image_size=32)
    model_cfg = {
        "model_name": model_name,
        "arch": arch,
        "defense_method": defense,
        "num_classes": num_classes,
        "image_size": 32,
    }
    return model, model_cfg

def load_rep_bundle(save_root, run_group, dataset_name, model_name, seed, checkpoint_tag="best"):
    if checkpoint_tag not in ("best", "last"):
        raise ValueError("checkpoint_tag must be 'best' or 'last'")

    rdir = run_dir(save_root, run_group, dataset_name, model_name, seed)
    meta_path = rdir / "meta.json"
    ckpt_path = rdir / f"{checkpoint_tag}_analysis.msgpack"

    if not meta_path.exists():
        raise FileNotFoundError(meta_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(ckpt_path)

    meta = load_json(meta_path)
    model, model_cfg = build_model(model_name, dataset_name)

    image_size = meta["dataset_cfg"]["image_size"]
    dummy = jnp.zeros((1, image_size, image_size, 3), dtype=jnp.float32)
    key = random.PRNGKey(0)

    init_vars = model.init({"params": key}, dummy, train=False)

    template = {
        "params": init_vars["params"],
        "batch_stats": init_vars.get("batch_stats", {}),
        "step": np.asarray(0, dtype=np.int32),
    }

    with open(ckpt_path, "rb") as f:
        payload = serialization.from_bytes(template, f.read())

    variables = {"params": payload["params"]}
    if payload["batch_stats"]:
        variables["batch_stats"] = payload["batch_stats"]

    layer_names = meta["layer_names"]

    def apply_logits(x, train=False):
        return model.apply(variables, x, train=train)

    def get_activations(x, train=False):
        logits, mut = model.apply(variables, x, train=train, mutable=["intermediates"])
        acts = {k: v[0] for k, v in mut["intermediates"].items()}
        return logits, acts

    def get_layer(x, layer_name, train=False):
        _, acts = get_activations(x, train=train)
        return acts[layer_name]

    def jvp_logits(x, v):
        f = lambda z: model.apply(variables, z, train=False)
        return jax.jvp(f, (x,), (v,))

    def jvp_layer(x, v, layer_name):
        def f(z):
            _, acts = get_activations(z, train=False)
            return acts[layer_name]
        return jax.jvp(f, (x,), (v,))

    bundle = {
        "run_dir": str(rdir),
        "meta": meta,
        "model_cfg": model_cfg,
        "model": model,
        "variables": variables,
        "step": int(payload["step"]),
        "layer_names": layer_names,
        "apply_logits": apply_logits,
        "get_activations": get_activations,
        "get_layer": get_layer,
        "jvp_logits": jvp_logits,
        "jvp_layer": jvp_layer,
    }
    return bundle

print("Unified loader ready.")

In [ ]:
# @title 3. Dataset helpers

def load_tfds_numpy(dataset_name, split):
    ds = tfds.load(dataset_name, split=split, batch_size=-1, as_supervised=True)
    x, y = tfds.as_numpy(ds)
    x = x.astype(np.float32) / 255.0
    y = y.astype(np.int64)

    mean = np.array(DATASET_CFG[dataset_name]["mean"], dtype=np.float32)
    std = np.array(DATASET_CFG[dataset_name]["std"], dtype=np.float32)

    x = (x - mean[None, None, None, :]) / std[None, None, None, :]
    return x, y

def normalize_from_pixel(x_pix, dataset_name):
    mean = np.array(DATASET_CFG[dataset_name]["mean"], dtype=np.float32)
    std = np.array(DATASET_CFG[dataset_name]["std"], dtype=np.float32)
    return (x_pix - mean[None, None, :]) / std[None, None, :]

def denormalize_to_pixel(x_norm, dataset_name):
    mean = np.array(DATASET_CFG[dataset_name]["mean"], dtype=np.float32)
    std = np.array(DATASET_CFG[dataset_name]["std"], dtype=np.float32)
    return np.clip(x_norm * std[None, None, :] + mean[None, None, :], 0.0, 1.0)

train_cache = os.path.join(ARRAY_DIR, f"{DATASET_NAME}_train_full.npz")
test_cache = os.path.join(ARRAY_DIR, f"{DATASET_NAME}_test_full.npz")

if os.path.exists(train_cache) and os.path.exists(test_cache):
    tr = np.load(train_cache)
    te = np.load(test_cache)
    X_train_full, y_train_full = tr["X"], tr["y"]
    X_test_full, y_test_full = te["X"], te["y"]
else:
    X_train_full, y_train_full = load_tfds_numpy(DATASET_NAME, "train")
    X_test_full, y_test_full = load_tfds_numpy(DATASET_NAME, "test")
    np.savez_compressed(train_cache, X=X_train_full, y=y_train_full)
    np.savez_compressed(test_cache, X=X_test_full, y=y_test_full)

print("Train:", X_train_full.shape, y_train_full.shape)
print("Test :", X_test_full.shape, y_test_full.shape)

mean = np.array(DATASET_CFG[DATASET_NAME]["mean"], dtype=np.float32)
std = np.array(DATASET_CFG[DATASET_NAME]["std"], dtype=np.float32)

In [ ]:
# @title 4. Build model bank manifest

model_rows = []

# standard models
for seed in STANDARD_SEEDS:
    model_rows.append({
        "group": "standard",
        "model_name": STANDARD_MODEL_NAME,
        "seed": seed,
        "run_group": CLEAN_RUN_GROUP,
        "save_root": SAVE_ROOT,
    })

# adversarial models
for adv_name in ADV_MODEL_NAMES:
    _, defense = parse_model_name(adv_name)
    for seed in ADV_SEEDS:
        model_rows.append({
            "group": defense,
            "model_name": adv_name,
            "seed": seed,
            "run_group": ADV_RUN_GROUP,
            "save_root": SAVE_ROOT,
        })

model_manifest = pd.DataFrame(model_rows)
model_manifest["dataset_name"] = DATASET_NAME
model_manifest["checkpoint_tag"] = CHECKPOINT_TAG

bundles = []
loaded_rows = []

for i, row in tqdm(model_manifest.iterrows(), total=len(model_manifest), desc="Loading model bank"):
    bundle = load_rep_bundle(
        save_root=row["save_root"],
        run_group=row["run_group"],
        dataset_name=row["dataset_name"],
        model_name=row["model_name"],
        seed=int(row["seed"]),
        checkpoint_tag=row["checkpoint_tag"],
    )
    bundles.append(bundle)

    loaded_rows.append({
        "model_idx": len(bundles) - 1,
        "group": row["group"],
        "model_name": row["model_name"],
        "seed": int(row["seed"]),
        "run_group": row["run_group"],
        "defense_method": bundle["model_cfg"]["defense_method"],
        "arch": bundle["model_cfg"]["arch"],
        "run_dir": bundle["run_dir"],
    })

bank_df = pd.DataFrame(loaded_rows)
bank_csv = os.path.join(TABLE_DIR, "model_bank_manifest.csv")
bank_df.to_csv(bank_csv, index=False)

print(bank_df)
print("\nSaved:", bank_csv)
print("Loaded models:", len(bundles))

In [ ]:
# @title 5. TPU helpers, logits, activations, clean benchmark selection

num_devices = jax.local_device_count()

def batch_iterator(X, batch_size=256):
    for i in range(0, len(X), batch_size):
        yield X[i:i+batch_size]

def pad_to_devices(x):
    n = x.shape[0]
    per_dev = int(np.ceil(n / num_devices))
    n_pad = per_dev * num_devices - n
    if n_pad > 0:
        x = np.concatenate([x, np.repeat(x[-1:], n_pad, axis=0)], axis=0)
    return x.reshape((num_devices, per_dev) + x.shape[1:]), n

@partial(jax.pmap, in_axes=(None, None, 0), static_broadcasted_argnums=(0,))
def compute_logits_chunk(apply_fn, variables, x_batch):
    return apply_fn(variables, x_batch, train=False)

@partial(jax.pmap, in_axes=(None, None, None, 0), static_broadcasted_argnums=(0, 1))
def compute_layer_chunk(apply_fn, layer_name, variables, x_batch):
    _, mut = apply_fn(variables, x_batch, train=False, mutable=["intermediates"])
    act = mut["intermediates"][layer_name][0]
    return act.reshape((act.shape[0], -1))

def batched_logits(bundle, X, batch_size=256):
    outs = []
    apply_fn = bundle["model"].apply
    variables = bundle["variables"]

    for xb in batch_iterator(X, batch_size=batch_size):
        x_sh, n_orig = pad_to_devices(xb)
        logits = compute_logits_chunk(apply_fn, variables, x_sh)
        logits = np.array(logits, dtype=np.float32).reshape(-1, np.array(logits).shape[-1])[:n_orig]
        outs.append(logits)

    return np.concatenate(outs, axis=0)

def extract_layer_features(bundle, X, layer_name, batch_size=256):
    outs = []
    apply_fn = bundle["model"].apply
    variables = bundle["variables"]

    for xb in batch_iterator(X, batch_size=batch_size):
        x_sh, n_orig = pad_to_devices(xb)
        feats = compute_layer_chunk(apply_fn, layer_name, variables, x_sh)
        feats = np.array(feats, dtype=np.float32).reshape(-1, np.array(feats).shape[-1])[:n_orig]
        outs.append(feats)

    return np.concatenate(outs, axis=0)

def true_class_margin(logits, y_true):
    N, C = logits.shape
    true_scores = logits[np.arange(N), y_true]
    mask = np.eye(C, dtype=bool)[y_true]
    other = np.where(mask, -np.inf, logits)
    other_max = np.max(other, axis=1)
    return true_scores - other_max

# deterministic data splits
train_perm = np.random.default_rng(GLOBAL_SEED).permutation(len(X_train_full))
test_perm = np.random.default_rng(GLOBAL_SEED + 1).permutation(len(X_test_full))

famlearn_idx = train_perm[:N_FAM_LEARN]
geom_idx = train_perm[N_FAM_LEARN:N_FAM_LEARN + N_GEOM]
cand_idx = test_perm[:N_BENCH_CANDIDATES]

X_famlearn = X_train_full[famlearn_idx]
X_geom = X_train_full[geom_idx]

X_cand = X_test_full[cand_idx]
y_cand = y_test_full[cand_idx]

# clean benchmark logits for candidate pool
clean_logits_candidates_path = os.path.join(ARRAY_DIR, "clean_logits_candidates.npy")
clean_correct_candidates_path = os.path.join(ARRAY_DIR, "clean_correct_candidates.npy")

if os.path.exists(clean_logits_candidates_path) and os.path.exists(clean_correct_candidates_path):
    clean_logits_cand = np.load(clean_logits_candidates_path)
    clean_correct_cand = np.load(clean_correct_candidates_path)
else:
    M = len(bundles)
    N = len(X_cand)
    C = DATASET_CFG[DATASET_NAME]["num_classes"]

    clean_logits_cand = np.zeros((M, N, C), dtype=np.float32)
    clean_correct_cand = np.zeros((M, N), dtype=bool)

    for m_idx, bundle in enumerate(tqdm(bundles, desc="Clean candidate logits")):
        logits = batched_logits(bundle, X_cand, batch_size=LOGIT_BATCH_SIZE)
        clean_logits_cand[m_idx] = logits
        clean_correct_cand[m_idx] = (np.argmax(logits, axis=1) == y_cand)

    np.save(clean_logits_candidates_path, clean_logits_cand)
    np.save(clean_correct_candidates_path, clean_correct_cand)

correct_counts = clean_correct_cand.sum(axis=0)
selected_mask = correct_counts >= MIN_CLEAN_CORRECT_COUNT
selected_indices_within_cand = np.where(selected_mask)[0]

if len(selected_indices_within_cand) < N_BENCH_IMAGES:
    raise RuntimeError(
        f"Only {len(selected_indices_within_cand)} candidate images meet the >= {MIN_CLEAN_CORRECT_COUNT} criterion; "
        f"need {N_BENCH_IMAGES}."
    )

selected_indices_within_cand = selected_indices_within_cand[:N_BENCH_IMAGES]
bench_idx = cand_idx[selected_indices_within_cand]

X_bench = X_cand[selected_indices_within_cand]
y_bench = y_cand[selected_indices_within_cand]

print("Family-learning split:", X_famlearn.shape)
print("Geometry split:", X_geom.shape)
print("Benchmark images:", X_bench.shape)

np.savez_compressed(
    os.path.join(ARRAY_DIR, "benchmark_selection.npz"),
    bench_idx=bench_idx,
    selected_indices_within_cand=selected_indices_within_cand,
    y_bench=y_bench,
    correct_counts=correct_counts[selected_indices_within_cand],
)

In [ ]:
# @title 6. Broad boundary-safe geometric flow bank

H = W = DATASET_CFG[DATASET_NAME]["image_size"]
yy, xx = np.meshgrid(np.arange(H, dtype=np.float32), np.arange(W, dtype=np.float32), indexing="ij")
sx = xx / (W - 1.0)
sy = yy / (H - 1.0)

def scalar_mode(u, v):
    return np.sin(np.pi * u * sx) * np.sin(np.pi * v * sy)

def build_broad_flow_bank():
    flows = []
    labels = []

    for (u, v) in MODE_LIST:
        phi = scalar_mode(u, v).astype(np.float32)

        fx = np.stack([phi, np.zeros_like(phi)], axis=-1)
        fy = np.stack([np.zeros_like(phi), phi], axis=-1)

        for f, name in [(fx, f"mode({u},{v})_dx"), (fy, f"mode({u},{v})_dy")]:
            rms = np.sqrt(np.mean(f[..., 0] ** 2 + f[..., 1] ** 2))
            f = f / (rms + 1e-12)   # 1 unit coefficient -> 1 pixel RMS displacement
            flows.append(f.astype(np.float32))
            labels.append(name)

    return np.stack(flows, axis=0), labels  # [72,H,W,2]

FLOW_BANK, FLOW_LABELS = build_broad_flow_bank()
np.save(os.path.join(ARRAY_DIR, "broad_flow_bank.npy"), FLOW_BANK)

def warp_with_flow_pixel(x_pix, flow, coeff):
    dx = coeff * flow[..., 0]
    dy = coeff * flow[..., 1]

    x_src = xx - dx
    y_src = yy - dy

    out = np.zeros_like(x_pix, dtype=np.float32)
    for c in range(3):
        out[..., c] = ndi.map_coordinates(
            x_pix[..., c],
            [y_src, x_src],
            order=1,
            mode="reflect",
        )
    return np.clip(out, 0.0, 1.0)

# quick visual sanity check
x0_pix = denormalize_to_pixel(X_famlearn[0], DATASET_NAME)

fig, axes = plt.subplots(2, 3, figsize=(8, 5))
axes = axes.ravel()

axes[0].imshow(x0_pix)
axes[0].set_title("Original")
axes[0].axis("off")

for k in range(1, 6):
    xw = warp_with_flow_pixel(x0_pix, FLOW_BANK[k - 1], coeff=1.0)
    axes[k].imshow(xw)
    axes[k].set_title(FLOW_LABELS[k - 1])
    axes[k].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# @title 7. Build base tangent banks

def build_base_tangent_bank(X_norm, flow_bank, fd_eps=0.5):
    """
    Returns [N, B, H, W, C] float16
    tangent_b(x) = (T_{+eps flow_b}(x) - T_{-eps flow_b}(x)) / (2 eps)
    """
    N = len(X_norm)
    B = flow_bank.shape[0]
    out = np.zeros((N, B, H, W, 3), dtype=np.float16)

    for n in tqdm(range(N), desc="Base tangent bank"):
        x_pix = denormalize_to_pixel(X_norm[n], DATASET_NAME)

        tangents = []
        for b in range(B):
            flow = flow_bank[b]

            xp = warp_with_flow_pixel(x_pix, flow, coeff=fd_eps)
            xm = warp_with_flow_pixel(x_pix, flow, coeff=-fd_eps)

            xp_n = normalize_from_pixel(xp, DATASET_NAME)
            xm_n = normalize_from_pixel(xm, DATASET_NAME)

            t = (xp_n - xm_n) / (2.0 * fd_eps)
            tangents.append(t.astype(np.float16))

        out[n] = np.stack(tangents, axis=0)

    return out

famlearn_tangent_path = os.path.join(ARRAY_DIR, "base_tangent_bank_famlearn.npy")
geom_tangent_path = os.path.join(ARRAY_DIR, "base_tangent_bank_geom.npy")

if not os.path.exists(famlearn_tangent_path):
    T_famlearn = build_base_tangent_bank(X_famlearn, FLOW_BANK, fd_eps=FD_EPS)
    np.save(famlearn_tangent_path, T_famlearn)
    del T_famlearn
    gc.collect()

if not os.path.exists(geom_tangent_path):
    T_geom = build_base_tangent_bank(X_geom, FLOW_BANK, fd_eps=FD_EPS)
    np.save(geom_tangent_path, T_geom)
    del T_geom
    gc.collect()

print("Saved:")
print(" ", famlearn_tangent_path)
print(" ", geom_tangent_path)

In [ ]:
# @title 8. Learn nested family basis A_64 and learned flow basis F_nat

family_basis_path = os.path.join(ARRAY_DIR, "learned_family_basis.npz")

def learn_family_from_tangent_covariance(base_tangent_path, k=64):
    T_bank = np.load(base_tangent_path, mmap_mode="r")
    N = T_bank.shape[0]
    B = T_bank.shape[1]

    C = np.zeros((B, B), dtype=np.float64)

    for n in tqdm(range(N), desc="Family covariance"):
        Tn = np.array(T_bank[n], dtype=np.float32).reshape(B, -1)  # [72, d]
        C += Tn @ Tn.T

    C /= N

    evals, evecs = np.linalg.eigh(C)
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    evecs = evecs[:, order]

    A = evecs[:, :k].astype(np.float32)   # [72, 64]
    return C.astype(np.float32), A, evals.astype(np.float32)

if os.path.exists(family_basis_path):
    dd = np.load(family_basis_path, allow_pickle=True)
    C_tan = dd["C_tan"]
    A64 = dd["A64"]
    evals_C = dd["evals_C"]
    F_nat64 = dd["F_nat64"]
else:
    C_tan, A64, evals_C = learn_family_from_tangent_covariance(famlearn_tangent_path, k=K_MAX)
    F_nat64 = np.tensordot(A64.T, FLOW_BANK, axes=(1, 0)).astype(np.float32)  # [64,H,W,2]

    np.savez_compressed(
        family_basis_path,
        C_tan=C_tan,
        A64=A64,
        evals_C=evals_C,
        F_nat64=F_nat64,
    )

print("A64 shape:", A64.shape)
print("F_nat64 shape:", F_nat64.shape)
print("Top eigenvalues:", evals_C[:10])

# visualize first few learned basis flows as warped examples
x0_pix = denormalize_to_pixel(X_famlearn[0], DATASET_NAME)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j in range(8):
    xw = warp_with_flow_pixel(x0_pix, F_nat64[j], coeff=0.75)
    ax = axes.ravel()[j]
    ax.imshow(xw)
    ax.set_title(f"F_nat {j+1}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# @title 9. Build learned tangent bank at Kmax and estimate G for selected layers

SELECTED_LAYERS_FOR_G = sorted(list(set([MAIN_LAYER] + LAYER_ABLATION_LAYERS)))

learned_geom_tangent_path = os.path.join(ARRAY_DIR, "learned_tangent_bank_geom_k64.npy")
G_full_path = os.path.join(ARRAY_DIR, "G_full_layers_k64.npy")
G_meta_csv = os.path.join(TABLE_DIR, "G_layer_manifest.csv")

def combine_base_tangents_with_A(base_tangent_path, A, out_path):
    T_bank = np.load(base_tangent_path, mmap_mode="r")
    N = T_bank.shape[0]
    K = A.shape[1]

    out = np.zeros((N, K, H, W, 3), dtype=np.float16)

    for n in tqdm(range(N), desc="Combining tangents with A"):
        Tn = np.array(T_bank[n], dtype=np.float32)   # [72,H,W,C]
        Un = np.tensordot(Tn, A, axes=(0, 0))        # [H,W,C,K]
        Un = np.transpose(Un, (3, 0, 1, 2))          # [K,H,W,C]
        out[n] = Un.astype(np.float16)

    np.save(out_path, out)

if not os.path.exists(learned_geom_tangent_path):
    combine_base_tangents_with_A(geom_tangent_path, A64, learned_geom_tangent_path)

@partial(jax.pmap, in_axes=(None, None, None, 0, 0), static_broadcasted_argnums=(0, 1))
def compute_H_chunk(apply_fn, layer_name, variables, x_batch, tangents_batch):
    def target_fn(x):
        _, mut = apply_fn(variables, x[None, ...], train=False, mutable=["intermediates"])
        act = mut["intermediates"][layer_name][0]
        return act.reshape(-1)

    def H_one(img, tangents):
        def single_jvp(t):
            _, tangent = jax.jvp(target_fn, (img,), (t,))
            return tangent
        V = jax.vmap(single_jvp)(tangents)  # [K,D]
        return V @ V.T                      # [K,K]

    return jax.vmap(H_one)(x_batch, tangents_batch)

def estimate_G_for_layer(bundle, X, learned_tangent_path, layer_name, batch_size_total=32):
    U_bank = np.load(learned_tangent_path, mmap_mode="r")
    N = len(X)
    G_sum = np.zeros((K_MAX, K_MAX), dtype=np.float64)

    apply_fn = bundle["model"].apply
    variables = bundle["variables"]

    for start in tqdm(range(0, N, batch_size_total), desc=f"G {layer_name}"):
        stop = min(start + batch_size_total, N)

        xb = X[start:stop]
        ub = np.array(U_bank[start:stop], dtype=np.float32)

        x_sh, n_orig = pad_to_devices(xb)
        u_sh, _ = pad_to_devices(ub)

        H_chunk = compute_H_chunk(apply_fn, layer_name, variables, x_sh, u_sh)
        H_chunk = np.array(H_chunk, dtype=np.float32).reshape(-1, K_MAX, K_MAX)[:n_orig]

        G_sum += H_chunk.sum(axis=0)

        del xb, ub, x_sh, u_sh, H_chunk
        gc.collect()

    return (G_sum / N).astype(np.float32)

if os.path.exists(G_full_path):
    G_full = np.load(G_full_path)  # [M,L,K,K]
else:
    M = len(bundles)
    L = len(SELECTED_LAYERS_FOR_G)
    G_full = np.zeros((M, L, K_MAX, K_MAX), dtype=np.float32)

    for m_idx, bundle in enumerate(tqdm(bundles, desc="Estimating G for all models/layers")):
        for l_idx, layer_name in enumerate(SELECTED_LAYERS_FOR_G):
            G_full[m_idx, l_idx] = estimate_G_for_layer(
                bundle,
                X_geom,
                learned_geom_tangent_path,
                layer_name,
                batch_size_total=GRAM_BATCH_SIZE_TOTAL,
            )
            gc.collect()

    np.save(G_full_path, G_full)

gmeta = pd.DataFrame({
    "layer_idx": np.arange(len(SELECTED_LAYERS_FOR_G)),
    "layer_name": SELECTED_LAYERS_FOR_G,
})
gmeta.to_csv(G_meta_csv, index=False)

print("G_full shape:", G_full.shape)
print("Saved:", G_full_path)
print(gmeta)

In [ ]:
# @title 10. Activation baselines (Linear/RBF CKA) for selected layers

CKA_DIST_PATH = os.path.join(ARRAY_DIR, "cka_distance_layers.npz")
RBF_BANDWIDTH_CSV = os.path.join(TABLE_DIR, "rbf_bandwidths.csv")

def center_gram(K):
    row_mean = K.mean(axis=1, keepdims=True)
    col_mean = K.mean(axis=0, keepdims=True)
    grand_mean = K.mean()
    return K - row_mean - col_mean + grand_mean

def linear_gram_from_acts(X):
    X = X.astype(np.float32)
    X = X - X.mean(axis=0, keepdims=True)
    return X @ X.T

def pairwise_sq_dists_from_acts(X):
    X = X.astype(np.float32)
    norms = np.sum(X * X, axis=1, keepdims=True)
    D2 = norms + norms.T - 2.0 * (X @ X.T)
    return np.maximum(D2, 0.0)

def median_heuristic_sigma_sq(D2):
    tri = D2[np.triu_indices(D2.shape[0], k=1)]
    tri = tri[tri > 0]
    if tri.size == 0:
        return 1.0
    return float(np.median(tri))

def rbf_centered_gram_from_acts(X):
    D2 = pairwise_sq_dists_from_acts(X)
    sigma_sq = median_heuristic_sigma_sq(D2)
    K = np.exp(-D2 / (2.0 * sigma_sq)).astype(np.float32)
    return center_gram(K), sigma_sq

def cka_from_centered_grams(Kc, Lc):
    num = np.sum(Kc * Lc)
    den = np.linalg.norm(Kc) * np.linalg.norm(Lc) + 1e-9
    return float(num / den)

if os.path.exists(CKA_DIST_PATH):
    dd = np.load(CKA_DIST_PATH, allow_pickle=True)
    linear_cka_dist = dd["linear_cka_dist"]   # [L,M,M]
    rbf_cka_dist = dd["rbf_cka_dist"]         # [L,M,M]
else:
    M = len(bundles)
    L = len(SELECTED_LAYERS_FOR_G)

    linear_cka_dist = np.zeros((L, M, M), dtype=np.float32)
    rbf_cka_dist = np.zeros((L, M, M), dtype=np.float32)

    rbf_rows = []

    for l_idx, layer_name in enumerate(SELECTED_LAYERS_FOR_G):
        print(f"\n>>> CKA for layer: {layer_name}")

        linear_grams = []
        rbf_grams = []

        for m_idx, bundle in enumerate(tqdm(bundles, desc=f"Layer {layer_name}")):
            X_feat = extract_layer_features(bundle, X_geom, layer_name=layer_name, batch_size=FEATURE_BATCH_SIZE)

            K_lin = linear_gram_from_acts(X_feat)
            K_rbf, sigma_sq = rbf_centered_gram_from_acts(X_feat)

            linear_grams.append(K_lin.astype(np.float32))
            rbf_grams.append(K_rbf.astype(np.float32))

            rbf_rows.append({
                "layer": layer_name,
                "model_idx": m_idx,
                "model_name": bank_df.iloc[m_idx]["model_name"],
                "seed": int(bank_df.iloc[m_idx]["seed"]),
                "sigma_sq": sigma_sq,
                "gamma": 1.0 / (2.0 * sigma_sq),
            })

            del X_feat, K_lin, K_rbf
            gc.collect()

        for i in range(M):
            for j in range(i + 1, M):
                s_lin = cka_from_centered_grams(linear_grams[i], linear_grams[j])
                s_rbf = cka_from_centered_grams(rbf_grams[i], rbf_grams[j])

                linear_cka_dist[l_idx, i, j] = 1.0 - s_lin
                linear_cka_dist[l_idx, j, i] = 1.0 - s_lin

                rbf_cka_dist[l_idx, i, j] = 1.0 - s_rbf
                rbf_cka_dist[l_idx, j, i] = 1.0 - s_rbf

    pd.DataFrame(rbf_rows).to_csv(RBF_BANDWIDTH_CSV, index=False)
    np.savez_compressed(
        CKA_DIST_PATH,
        linear_cka_dist=linear_cka_dist,
        rbf_cka_dist=rbf_cka_dist,
    )

print("linear_cka_dist shape:", linear_cka_dist.shape)
print("rbf_cka_dist shape:", rbf_cka_dist.shape)

In [ ]:
# @title 11. SPD metric helpers and clean benchmark margins

def spd_lift_trace_scaled(G, eps_reg=1e-4):
    k = G.shape[0]
    tr = float(np.trace(G))
    scale = tr / k if tr > 0 else 1.0
    return G + (eps_reg * scale) * np.eye(k, dtype=G.dtype)

def matrix_log_spd(A):
    evals, evecs = np.linalg.eigh(A)
    evals = np.maximum(evals, 1e-12)
    return evecs @ np.diag(np.log(evals)) @ evecs.T

def sras_distance(G1, G2, eps_reg=1e-4):
    A = spd_lift_trace_scaled(G1, eps_reg=eps_reg)
    B = spd_lift_trace_scaled(G2, eps_reg=eps_reg)

    evals, evecs = np.linalg.eigh(A)
    A_inv_sqrt = evecs @ np.diag(1.0 / np.sqrt(np.maximum(evals, 1e-12))) @ evecs.T
    mid = A_inv_sqrt @ B @ A_inv_sqrt
    evals_mid = np.linalg.eigvalsh(mid)
    return float(np.sqrt(np.sum(np.log(np.maximum(evals_mid, 1e-12)) ** 2)) / np.sqrt(G1.shape[0]))

def loge_distance(G1, G2, eps_reg=1e-4):
    A = spd_lift_trace_scaled(G1, eps_reg=eps_reg)
    B = spd_lift_trace_scaled(G2, eps_reg=eps_reg)
    return float(np.linalg.norm(matrix_log_spd(A) - matrix_log_spd(B), ord="fro") / np.sqrt(G1.shape[0]))

# clean benchmark logits/margins for all models
clean_bench_path = os.path.join(ARRAY_DIR, "clean_bench_logits_margins.npz")

if os.path.exists(clean_bench_path):
    cc = np.load(clean_bench_path, allow_pickle=True)
    clean_logits = cc["clean_logits"]
    clean_correct = cc["clean_correct"]
    clean_margins = cc["clean_margins"]
else:
    M = len(bundles)
    N = len(X_bench)
    C = DATASET_CFG[DATASET_NAME]["num_classes"]

    clean_logits = np.zeros((M, N, C), dtype=np.float32)
    clean_correct = np.zeros((M, N), dtype=bool)
    clean_margins = np.zeros((M, N), dtype=np.float32)

    for m_idx, bundle in enumerate(tqdm(bundles, desc="Clean benchmark logits")):
        logits = batched_logits(bundle, X_bench, batch_size=LOGIT_BATCH_SIZE)
        clean_logits[m_idx] = logits
        clean_correct[m_idx] = (np.argmax(logits, axis=1) == y_bench)
        clean_margins[m_idx] = true_class_margin(logits, y_bench)

    np.savez_compressed(
        clean_bench_path,
        clean_logits=clean_logits,
        clean_correct=clean_correct,
        clean_margins=clean_margins,
    )

print("Mean clean-correct rate on benchmark images:", clean_correct.mean())

In [ ]:
# @title 12. Pairwise comparison specifications and balanced discovery/test splits

COMPARISONS = [
    ("standard", "pgd"),
    ("standard", "trades"),
    ("standard", "mart"),
    ("pgd", "trades"),
    ("pgd", "mart"),
    ("trades", "mart"),
]

split_specs_json = os.path.join(TABLE_DIR, "comparison_split_specs.json")

def group_indices(group_name):
    return bank_df.index[bank_df["group"] == group_name].tolist()

comparison_split_specs = []

for comp_id, (group_a, group_b) in enumerate(COMPARISONS):
    idx_a_all = group_indices(group_a)
    idx_b_all = group_indices(group_b)

    n_match = min(len(idx_a_all), len(idx_b_all))
    n_disc = max(2, int(np.ceil(DISC_FRAC * n_match)))
    n_test = n_match - n_disc

    if n_test < 2:
        raise RuntimeError(f"Too few held-out models for comparison {group_a} vs {group_b}.")

    for split_id in range(N_SPLITS):
        rng_split = np.random.default_rng(GLOBAL_SEED + 1000 * comp_id + split_id)

        idx_a_sub = sorted(rng_split.choice(idx_a_all, size=n_match, replace=False).tolist())
        idx_b_sub = sorted(rng_split.choice(idx_b_all, size=n_match, replace=False).tolist())

        disc_a = sorted(rng_split.choice(idx_a_sub, size=n_disc, replace=False).tolist())
        disc_b = sorted(rng_split.choice(idx_b_sub, size=n_disc, replace=False).tolist())

        test_a = sorted([i for i in idx_a_sub if i not in disc_a])
        test_b = sorted([i for i in idx_b_sub if i not in disc_b])

        comparison_split_specs.append({
            "comparison_id": comp_id,
            "comparison_name": f"{group_a}_vs_{group_b}",
            "group_a": group_a,
            "group_b": group_b,
            "split_id": split_id,
            "n_match": n_match,
            "n_disc": n_disc,
            "n_test": n_test,
            "disc_a": disc_a,
            "disc_b": disc_b,
            "test_a": test_a,
            "test_b": test_b,
        })

with open(split_specs_json, "w") as f:
    json.dump(comparison_split_specs, f, indent=2)

print("Saved:", split_specs_json)
print(comparison_split_specs[0])

In [ ]:
# @title 13. Probe derivation helpers

def derive_contrast_probes(deltaG, probes_per_side=2):
    evals, evecs = np.linalg.eigh(deltaG)
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    evecs = evecs[:, order]

    pos_idx = np.where(evals > 0)[0]
    neg_idx = np.where(evals < 0)[0]

    if len(pos_idx) >= probes_per_side:
        pos_dirs = evecs[:, pos_idx[:probes_per_side]].T
    else:
        pos_dirs = evecs[:, :probes_per_side].T

    if len(neg_idx) >= probes_per_side:
        neg_dirs = evecs[:, neg_idx[-probes_per_side:]].T
    else:
        neg_dirs = evecs[:, -probes_per_side:].T

    return {
        "pos_dirs": pos_dirs.astype(np.float32),
        "neg_dirs": neg_dirs.astype(np.float32),
        "evals": evals.astype(np.float32),
    }

def derive_random_contrast_probes(deltaG, k, n_random=128, probes_per_side=2, seed=0):
    rng_local = np.random.default_rng(seed)
    Z = rng_local.standard_normal((n_random, k)).astype(np.float32)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12

    scores = np.einsum("nk,kl,nl->n", Z, deltaG, Z)

    pos_idx = np.argsort(scores)[::-1][:probes_per_side]
    neg_idx = np.argsort(scores)[:probes_per_side]

    return {
        "pos_dirs": Z[pos_idx].astype(np.float32),
        "neg_dirs": Z[neg_idx].astype(np.float32),
        "scores": scores.astype(np.float32),
    }

def derive_pooled_sensitivity_probes(Gbar, deltaG, top_pool=12, probes_per_side=2):
    evals, evecs = np.linalg.eigh(Gbar)
    order = np.argsort(evals)[::-1]
    evecs = evecs[:, order]

    Z = evecs[:, :top_pool].T.astype(np.float32)
    scores = np.einsum("nk,kl,nl->n", Z, deltaG, Z)

    pos_idx = np.argsort(scores)[::-1][:probes_per_side]
    neg_idx = np.argsort(scores)[:probes_per_side]

    return {
        "pos_dirs": Z[pos_idx].astype(np.float32),
        "neg_dirs": Z[neg_idx].astype(np.float32),
        "scores": scores.astype(np.float32),
    }

def derive_label_permutation_probes(G_a_disc, G_b_disc, probes_per_side=2, seed=0):
    rng_local = np.random.default_rng(seed)

    X = np.concatenate([G_a_disc, G_b_disc], axis=0)
    labels = np.array([0] * len(G_a_disc) + [1] * len(G_b_disc))
    perm = rng_local.permutation(len(labels))
    labels_perm = labels[perm]

    G_a_fake = X[labels_perm == 0].mean(axis=0)
    G_b_fake = X[labels_perm == 1].mean(axis=0)
    deltaG_fake = G_a_fake - G_b_fake

    out = derive_contrast_probes(deltaG_fake, probes_per_side=probes_per_side)
    out["deltaG_fake"] = deltaG_fake.astype(np.float32)
    return out

def combine_family_dir_to_flow(z, F_nat_k):
    """
    z: [k]
    F_nat_k: [k,H,W,2]
    returns normalized combined flow [H,W,2] with RMS = 1 pixel
    """
    flow = np.tensordot(z, F_nat_k, axes=(0, 0)).astype(np.float32)
    rms = np.sqrt(np.mean(flow[..., 0] ** 2 + flow[..., 1] ** 2))
    flow = flow / (rms + 1e-12)
    return flow

def probe_dict_to_flows(probe_dict, F_nat_k):
    pos_flows = np.stack([combine_family_dir_to_flow(z, F_nat_k) for z in probe_dict["pos_dirs"]], axis=0)
    neg_flows = np.stack([combine_family_dir_to_flow(z, F_nat_k) for z in probe_dict["neg_dirs"]], axis=0)
    return {
        "pos_flows": pos_flows.astype(np.float32),
        "neg_flows": neg_flows.astype(np.float32),
    }

In [ ]:
# @title 14. Main diagnostic-probe benchmark on act_pre_logits for K = 64,32,16,8

diag_split_summary_csv = os.path.join(TABLE_DIR, "diag_probe_split_summary_main.csv")
diag_image_sep_csv = os.path.join(TABLE_DIR, "diag_probe_image_sep_main.csv")
diag_model_score_csv = os.path.join(TABLE_DIR, "diag_probe_model_scores_main.csv")
diag_probe_meta_csv = os.path.join(TABLE_DIR, "diag_probe_meta_main.csv")

layer_idx_main = SELECTED_LAYERS_FOR_G.index(MAIN_LAYER)

def build_transformed_stack_for_image(x_norm, pos_flows, neg_flows):
    """
    returns [2P, 2 signs, 2 amps, H, W, C]
    """
    x_pix = denormalize_to_pixel(x_norm, DATASET_NAME)
    probe_flows = np.concatenate([pos_flows, neg_flows], axis=0)
    n_probes = probe_flows.shape[0]

    out = np.zeros((n_probes, 2, len(FLOW_RMS_AMPS), H, W, 3), dtype=np.float32)
    signs = [+1.0, -1.0]

    for p_idx in range(n_probes):
        flow = probe_flows[p_idx]
        for s_idx, sgn in enumerate(signs):
            for a_idx, amp in enumerate(FLOW_RMS_AMPS):
                xt = warp_with_flow_pixel(x_pix, flow, coeff=sgn * amp)
                out[p_idx, s_idx, a_idx] = normalize_from_pixel(xt, DATASET_NAME)

    return out

def model_image_regime_score(bundle, transformed_stack, y_true, clean_margin):
    """
    transformed_stack: [2P,2,2,H,W,C]
    returns regime_score, per_probe_responses [2P]
    """
    n_probes = transformed_stack.shape[0]
    flat = transformed_stack.reshape(n_probes * 2 * len(FLOW_RMS_AMPS), H, W, 3)

    logits = batched_logits(bundle, flat, batch_size=LOGIT_BATCH_SIZE)
    logits = logits.reshape(n_probes, 2, len(FLOW_RMS_AMPS), -1)

    probe_resp = np.zeros(n_probes, dtype=np.float32)

    for p_idx in range(n_probes):
        vals = []
        for s_idx in range(2):
            for a_idx in range(len(FLOW_RMS_AMPS)):
                lt = logits[p_idx, s_idx, a_idx]
                true_score = lt[y_true]
                other = np.delete(lt, y_true)
                other_max = np.max(other)
                margin = true_score - other_max
                vals.append(clean_margin - margin)
        probe_resp[p_idx] = np.mean(vals)

    P = n_probes // 2
    score = probe_resp[:P].mean() - probe_resp[P:].mean()
    return float(score), probe_resp

diag_split_rows = []
diag_image_rows = []
diag_model_rows = []
diag_probe_rows = []

for k in K_LIST:
    print(f"\n=== Running diagnostic probes for K={k} ===")
    Gk_all = G_full[:, layer_idx_main, :k, :k]
    F_nat_k = F_nat64[:k]

    for spec in tqdm(comparison_split_specs, desc=f"Comparisons K={k}"):
        comp_name = spec["comparison_name"]
        group_a = spec["group_a"]
        group_b = spec["group_b"]
        split_id = spec["split_id"]

        disc_a = spec["disc_a"]
        disc_b = spec["disc_b"]
        test_a = spec["test_a"]
        test_b = spec["test_b"]
        test_all = test_a + test_b

        G_a_disc = Gk_all[disc_a]
        G_b_disc = Gk_all[disc_b]

        G_a_mean = G_a_disc.mean(axis=0)
        G_b_mean = G_b_disc.mean(axis=0)
        deltaG = G_a_mean - G_b_mean
        Gbar = np.concatenate([G_a_disc, G_b_disc], axis=0).mean(axis=0)

        probe_sets = {
            "contrast": derive_contrast_probes(deltaG, probes_per_side=PROBES_PER_SIDE),
            "random_contrast": derive_random_contrast_probes(
                deltaG,
                k=k,
                n_random=N_RANDOM_CAND,
                probes_per_side=PROBES_PER_SIDE,
                seed=GLOBAL_SEED + 10000 + hash((comp_name, split_id, k)) % 1000000,
            ),
            "pooled_sensitivity": derive_pooled_sensitivity_probes(
                Gbar,
                deltaG,
                top_pool=min(TOP_POOLED_EIGS, k),
                probes_per_side=PROBES_PER_SIDE,
            ),
            "label_permutation": derive_label_permutation_probes(
                G_a_disc,
                G_b_disc,
                probes_per_side=PROBES_PER_SIDE,
                seed=GLOBAL_SEED + 20000 + hash((comp_name, split_id, k)) % 1000000,
            ),
        }

        for probe_type, probe_dict in probe_sets.items():
            flows = probe_dict_to_flows(probe_dict, F_nat_k)
            pos_flows = flows["pos_flows"]
            neg_flows = flows["neg_flows"]

            for side_name, dirs in [("pos", probe_dict["pos_dirs"]), ("neg", probe_dict["neg_dirs"])]:
                for i, z in enumerate(dirs):
                    diag_probe_rows.append({
                        "comparison_name": comp_name,
                        "group_a": group_a,
                        "group_b": group_b,
                        "split_id": split_id,
                        "k": k,
                        "probe_type": probe_type,
                        "side": side_name,
                        "probe_idx": i,
                        "z_norm": float(np.linalg.norm(z)),
                    })

            per_model_image_score = {}

            for model_idx in test_all:
                bundle = bundles[model_idx]

                for img_idx in range(len(X_bench)):
                    if not clean_correct[model_idx, img_idx]:
                        continue

                    transformed_stack = build_transformed_stack_for_image(
                        X_bench[img_idx], pos_flows, neg_flows
                    )

                    score, probe_resp = model_image_regime_score(
                        bundle,
                        transformed_stack,
                        y_bench[img_idx],
                        clean_margins[model_idx, img_idx],
                    )

                    per_model_image_score[(model_idx, img_idx)] = score

                    diag_model_rows.append({
                        "comparison_name": comp_name,
                        "group_a": group_a,
                        "group_b": group_b,
                        "split_id": split_id,
                        "k": k,
                        "probe_type": probe_type,
                        "model_idx": model_idx,
                        "model_group": bank_df.iloc[model_idx]["group"],
                        "model_name": bank_df.iloc[model_idx]["model_name"],
                        "seed": int(bank_df.iloc[model_idx]["seed"]),
                        "img_idx": img_idx,
                        "score": score,
                    })

            img_sep_vals = []
            for img_idx in range(len(X_bench)):
                a_scores = [
                    per_model_image_score[(m, img_idx)]
                    for m in test_a
                    if (m, img_idx) in per_model_image_score
                ]
                b_scores = [
                    per_model_image_score[(m, img_idx)]
                    for m in test_b
                    if (m, img_idx) in per_model_image_score
                ]

                if len(a_scores) < 1 or len(b_scores) < 1:
                    continue

                sep = float(np.mean(a_scores) - np.mean(b_scores))
                img_sep_vals.append(sep)

                diag_image_rows.append({
                    "comparison_name": comp_name,
                    "group_a": group_a,
                    "group_b": group_b,
                    "split_id": split_id,
                    "k": k,
                    "probe_type": probe_type,
                    "img_idx": img_idx,
                    "image_sep": sep,
                    "n_group_a_models": len(a_scores),
                    "n_group_b_models": len(b_scores),
                })

            # model-level AUC from model mean score
            model_means = []
            model_labels = []

            for m in test_all:
                vals = [
                    per_model_image_score[(m, img_idx)]
                    for img_idx in range(len(X_bench))
                    if (m, img_idx) in per_model_image_score
                ]
                if len(vals) == 0:
                    continue

                model_means.append(np.mean(vals))
                model_labels.append(1 if bank_df.iloc[m]["group"] == group_a else 0)

            auc = np.nan
            if len(np.unique(model_labels)) == 2:
                auc = roc_auc_score(model_labels, model_means)

            diag_split_rows.append({
                "comparison_name": comp_name,
                "group_a": group_a,
                "group_b": group_b,
                "split_id": split_id,
                "k": k,
                "probe_type": probe_type,
                "mean_image_separation": np.mean(img_sep_vals) if len(img_sep_vals) > 0 else np.nan,
                "model_auc": auc,
                "n_valid_images": len(img_sep_vals),
                "n_test_models": len(test_all),
            })

pd.DataFrame(diag_split_rows).to_csv(diag_split_summary_csv, index=False)
pd.DataFrame(diag_image_rows).to_csv(diag_image_sep_csv, index=False)
pd.DataFrame(diag_model_rows).to_csv(diag_model_score_csv, index=False)
pd.DataFrame(diag_probe_rows).to_csv(diag_probe_meta_csv, index=False)

print("Saved:")
print(" ", diag_split_summary_csv)
print(" ", diag_image_sep_csv)
print(" ", diag_model_score_csv)
print(" ", diag_probe_meta_csv)

display(pd.read_csv(diag_split_summary_csv).head())

In [ ]:
# @title 15. Plain metric comparison: held-out group separation with S-RAS / LogE / CKA

plain_metric_csv = os.path.join(TABLE_DIR, "plain_metric_split_summary.csv")

plain_rows = []

for layer_name in SELECTED_LAYERS_FOR_G:
    l_idx = SELECTED_LAYERS_FOR_G.index(layer_name)

    for k in K_LIST:
        Gk_all = G_full[:, l_idx, :k, :k]

        # precompute metric distance matrices for this layer/k
        M = len(bundles)

        sras_dist = np.zeros((M, M), dtype=np.float32)
        loge_dist = np.zeros((M, M), dtype=np.float32)

        for i in range(M):
            for j in range(i + 1, M):
                ds = sras_distance(Gk_all[i], Gk_all[j])
                dl = loge_distance(Gk_all[i], Gk_all[j])

                sras_dist[i, j] = ds
                sras_dist[j, i] = ds

                loge_dist[i, j] = dl
                loge_dist[j, i] = dl

        lin_dist = linear_cka_dist[l_idx]
        rbf_dist = rbf_cka_dist[l_idx]

        for spec in comparison_split_specs:
            comp_name = spec["comparison_name"]
            group_a = spec["group_a"]
            group_b = spec["group_b"]
            split_id = spec["split_id"]

            disc_a = spec["disc_a"]
            disc_b = spec["disc_b"]
            test_a = spec["test_a"]
            test_b = spec["test_b"]
            test_all = test_a + test_b

            for metric_name, D in [
                ("S-RAS", sras_dist),
                ("LogE", loge_dist),
                ("Linear CKA", lin_dist),
                ("RBF CKA", rbf_dist),
            ]:
                model_scores = []
                model_labels = []

                for m in test_all:
                    d_to_a = np.mean([D[m, j] for j in disc_a])
                    d_to_b = np.mean([D[m, j] for j in disc_b])

                    # positive -> closer to group_a
                    score = d_to_b - d_to_a
                    model_scores.append(score)
                    model_labels.append(1 if bank_df.iloc[m]["group"] == group_a else 0)

                auc = roc_auc_score(model_labels, model_scores)

                mean_a = np.mean([s for s, y in zip(model_scores, model_labels) if y == 1])
                mean_b = np.mean([s for s, y in zip(model_scores, model_labels) if y == 0])
                separation = mean_a - mean_b

                plain_rows.append({
                    "comparison_name": comp_name,
                    "group_a": group_a,
                    "group_b": group_b,
                    "split_id": split_id,
                    "layer": layer_name,
                    "k": k,
                    "metric": metric_name,
                    "mean_score_separation": separation,
                    "model_auc": auc,
                    "n_test_models": len(test_all),
                })

pd.DataFrame(plain_rows).to_csv(plain_metric_csv, index=False)

print("Saved:", plain_metric_csv)
display(pd.read_csv(plain_metric_csv).head())

In [ ]:
# @title 16. Statistical testing and summary tables

diag_split_df = pd.read_csv(diag_split_summary_csv)
plain_df = pd.read_csv(plain_metric_csv)

diag_summary_csv = os.path.join(TABLE_DIR, "diag_probe_summary.csv")
diag_stats_csv = os.path.join(TABLE_DIR, "diag_probe_stats.csv")
plain_summary_csv = os.path.join(TABLE_DIR, "plain_metric_summary.csv")
plain_stats_csv = os.path.join(TABLE_DIR, "plain_metric_stats.csv")
diag_boot_csv = os.path.join(TABLE_DIR, "diag_probe_bootstrap.csv")

# ---------------------------
# Diagnostic probe summaries
# ---------------------------
diag_summary = (
    diag_split_df
    .groupby(["comparison_name", "k", "probe_type"], as_index=False)
    .agg(
        mean_image_separation=("mean_image_separation", "mean"),
        mean_model_auc=("model_auc", "mean"),
        n_splits=("split_id", "nunique")
    )
)
diag_summary.to_csv(diag_summary_csv, index=False)

# paired Wilcoxon across split means
diag_stats_rows = []
for comp_name in diag_split_df["comparison_name"].unique():
    for k in K_LIST:
        sub = diag_split_df[
            (diag_split_df["comparison_name"] == comp_name) &
            (diag_split_df["k"] == k)
        ]

        contrast = sub[sub["probe_type"] == "contrast"].sort_values("split_id")
        for baseline in ["random_contrast", "pooled_sensitivity", "label_permutation"]:
            base = sub[sub["probe_type"] == baseline].sort_values("split_id")

            x = contrast["mean_image_separation"].values
            y = base["mean_image_separation"].values

            if len(x) == len(y) and len(x) > 0:
                stat, p = stats.wilcoxon(x, y, alternative="greater")
            else:
                stat, p = np.nan, np.nan

            diag_stats_rows.append({
                "comparison_name": comp_name,
                "k": k,
                "test": f"contrast_gt_{baseline}",
                "wilcoxon_stat": stat,
                "pvalue": p,
                "mean_contrast": np.mean(x),
                "mean_baseline": np.mean(y),
            })

diag_stats_df = pd.DataFrame(diag_stats_rows)
diag_stats_df.to_csv(diag_stats_csv, index=False)

# bootstrap over benchmark images for main diagnostic results
diag_image_df = pd.read_csv(diag_image_sep_csv)
boot_rows = []

for b in tqdm(range(N_BOOT), desc="Bootstrap diagnostic results"):
    rng_boot = np.random.default_rng(GLOBAL_SEED + 999 + b)

    for comp_name in diag_image_df["comparison_name"].unique():
        for k in K_LIST:
            for probe_type in ["contrast", "random_contrast", "pooled_sensitivity", "label_permutation"]:
                split_means = []
                sub = diag_image_df[
                    (diag_image_df["comparison_name"] == comp_name) &
                    (diag_image_df["k"] == k) &
                    (diag_image_df["probe_type"] == probe_type)
                ]

                for split_id in sub["split_id"].unique():
                    vals = sub[sub["split_id"] == split_id]["image_sep"].values
                    if len(vals) == 0:
                        continue
                    samp = rng_boot.choice(vals, size=len(vals), replace=True)
                    split_means.append(np.mean(samp))

                boot_rows.append({
                    "boot": b,
                    "comparison_name": comp_name,
                    "k": k,
                    "probe_type": probe_type,
                    "quantity": "mean_image_separation",
                    "value": np.mean(split_means) if len(split_means) > 0 else np.nan,
                })

diag_boot_df = pd.DataFrame(boot_rows)
diag_boot_df.to_csv(diag_boot_csv, index=False)

# ---------------------------
# Plain metric summaries
# ---------------------------
plain_summary = (
    plain_df
    .groupby(["comparison_name", "layer", "k", "metric"], as_index=False)
    .agg(
        mean_score_separation=("mean_score_separation", "mean"),
        mean_model_auc=("model_auc", "mean"),
        n_splits=("split_id", "nunique"),
    )
)
plain_summary.to_csv(plain_summary_csv, index=False)

plain_stats_rows = []
for comp_name in plain_df["comparison_name"].unique():
    for layer in plain_df["layer"].unique():
        for k in K_LIST:
            sub = plain_df[
                (plain_df["comparison_name"] == comp_name) &
                (plain_df["layer"] == layer) &
                (plain_df["k"] == k)
            ]

            ref = sub[sub["metric"] == "S-RAS"].sort_values("split_id")
            for baseline in ["LogE", "Linear CKA", "RBF CKA"]:
                base = sub[sub["metric"] == baseline].sort_values("split_id")

                x = ref["model_auc"].values
                y = base["model_auc"].values

                if len(x) == len(y) and len(x) > 0:
                    stat, p = stats.wilcoxon(x, y, alternative="greater")
                else:
                    stat, p = np.nan, np.nan

                plain_stats_rows.append({
                    "comparison_name": comp_name,
                    "layer": layer,
                    "k": k,
                    "test": f"S-RAS_gt_{baseline}",
                    "wilcoxon_stat": stat,
                    "pvalue": p,
                    "mean_sras_auc": np.mean(x),
                    "mean_base_auc": np.mean(y),
                })

plain_stats_df = pd.DataFrame(plain_stats_rows)
plain_stats_df.to_csv(plain_stats_csv, index=False)

print("Saved:")
print(" ", diag_summary_csv)
print(" ", diag_stats_csv)
print(" ", diag_boot_csv)
print(" ", plain_summary_csv)
print(" ", plain_stats_csv)

display(diag_summary.head())
display(diag_stats_df.head())
display(plain_summary.head())
display(plain_stats_df.head())

In [ ]:
# @title 17. Layer ablation for K=16

# Reuse the already estimated G_full; rerun only the split-level computations at K=16 for selected layers.
layer_ablation_csv = os.path.join(TABLE_DIR, "layer_ablation_diag_probe_summary.csv")

layer_rows = []

for layer_name in LAYER_ABLATION_LAYERS:
    print(f"\n=== Layer ablation: {layer_name} ===")
    l_idx = SELECTED_LAYERS_FOR_G.index(layer_name)
    k = LAYER_ABLATION_K
    Gk_all = G_full[:, l_idx, :k, :k]
    F_nat_k = F_nat64[:k]

    for spec in tqdm(comparison_split_specs, desc=f"{layer_name}"):
        comp_name = spec["comparison_name"]
        group_a = spec["group_a"]
        group_b = spec["group_b"]
        split_id = spec["split_id"]

        disc_a = spec["disc_a"]
        disc_b = spec["disc_b"]
        test_a = spec["test_a"]
        test_b = spec["test_b"]
        test_all = test_a + test_b

        G_a_disc = Gk_all[disc_a]
        G_b_disc = Gk_all[disc_b]

        G_a_mean = G_a_disc.mean(axis=0)
        G_b_mean = G_b_disc.mean(axis=0)
        deltaG = G_a_mean - G_b_mean
        Gbar = np.concatenate([G_a_disc, G_b_disc], axis=0).mean(axis=0)

        probe_sets = {
            "contrast": derive_contrast_probes(deltaG, probes_per_side=PROBES_PER_SIDE),
            "random_contrast": derive_random_contrast_probes(
                deltaG,
                k=k,
                n_random=N_RANDOM_CAND,
                probes_per_side=PROBES_PER_SIDE,
                seed=GLOBAL_SEED + 30000 + hash((comp_name, split_id, layer_name)) % 1000000,
            ),
            "pooled_sensitivity": derive_pooled_sensitivity_probes(
                Gbar,
                deltaG,
                top_pool=min(TOP_POOLED_EIGS, k),
                probes_per_side=PROBES_PER_SIDE,
            ),
            "label_permutation": derive_label_permutation_probes(
                G_a_disc,
                G_b_disc,
                probes_per_side=PROBES_PER_SIDE,
                seed=GLOBAL_SEED + 40000 + hash((comp_name, split_id, layer_name)) % 1000000,
            ),
        }

        for probe_type, probe_dict in probe_sets.items():
            flows = probe_dict_to_flows(probe_dict, F_nat_k)
            pos_flows = flows["pos_flows"]
            neg_flows = flows["neg_flows"]

            per_model_image_score = {}

            for model_idx in test_all:
                bundle = bundles[model_idx]

                for img_idx in range(len(X_bench)):
                    if not clean_correct[model_idx, img_idx]:
                        continue

                    transformed_stack = build_transformed_stack_for_image(
                        X_bench[img_idx], pos_flows, neg_flows
                    )
                    score, _ = model_image_regime_score(
                        bundle,
                        transformed_stack,
                        y_bench[img_idx],
                        clean_margins[model_idx, img_idx],
                    )
                    per_model_image_score[(model_idx, img_idx)] = score

            img_sep_vals = []
            for img_idx in range(len(X_bench)):
                a_scores = [
                    per_model_image_score[(m, img_idx)]
                    for m in test_a
                    if (m, img_idx) in per_model_image_score
                ]
                b_scores = [
                    per_model_image_score[(m, img_idx)]
                    for m in test_b
                    if (m, img_idx) in per_model_image_score
                ]
                if len(a_scores) < 1 or len(b_scores) < 1:
                    continue
                sep = float(np.mean(a_scores) - np.mean(b_scores))
                img_sep_vals.append(sep)

            layer_rows.append({
                "comparison_name": comp_name,
                "split_id": split_id,
                "layer": layer_name,
                "probe_type": probe_type,
                "k": k,
                "mean_image_separation": np.mean(img_sep_vals) if len(img_sep_vals) > 0 else np.nan,
            })

pd.DataFrame(layer_rows).to_csv(layer_ablation_csv, index=False)
print("Saved:", layer_ablation_csv)

layer_df = pd.read_csv(layer_ablation_csv)
display(layer_df.head())

# plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, comp in zip(axes, ["standard_vs_pgd", "standard_vs_trades", "standard_vs_mart"]):
    for probe_type in ["contrast", "random_contrast", "pooled_sensitivity", "label_permutation"]:
        vals = []
        for layer_name in LAYER_ABLATION_LAYERS:
            sub = layer_df[
                (layer_df["comparison_name"] == comp) &
                (layer_df["layer"] == layer_name) &
                (layer_df["probe_type"] == probe_type)
            ]
            vals.append(sub["mean_image_separation"].mean())

        ax.plot(np.arange(len(LAYER_ABLATION_LAYERS)), vals, marker="o", label=probe_type)

    ax.set_title(comp)
    ax.set_xticks(np.arange(len(LAYER_ABLATION_LAYERS)))
    ax.set_xticklabels(LAYER_ABLATION_LAYERS, rotation=20, ha="right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Held-out image separation")
axes[0].legend(frameon=False, fontsize=10)
fig.suptitle(f"Layer ablation (K={LAYER_ABLATION_K})", y=1.03)
fig.tight_layout()
save_figure_both(fig, "panel_E_layer_ablation_k16")